# Step 2 · Python models · MCI

Endpoint: **MCI (CU → MCI)**. Fits the three Python models on the shared month-48 landmark inputs and writes their
predictions in the common format used by every model. Metrics, tables and figures are produced by
`04_evaluate_and_plot.ipynb`, which scores all models with the same code.

| Model | Input | History conditions |
|---|---|---|
| Landmark Cox (ridge) | 4 baseline features + per repeated measure: latest value, slope/year, mean − latest; months since last visit | first visit, latest 1, 2, 3, all; baseline features only |
| DeepSurv | same columns as Landmark Cox | first visit, latest 1, 2, 3, all |
| Dynamic-DeepHit (landmark adaptation) | ordered visit values, masks, time to month 48, gaps between visits; 4 baseline features | first visit, latest 1, 2, 3, all |

Visits are selected **backward from month 48** (latest 1, 2, 3 or all); the first-visit condition uses the earliest visit.
Hyperparameters are tuned on an inner 25% split of each training set (NACC outer folds), then refitted.
The final full-NACC model is applied unchanged to AIBL.

**Requires:** `01_prepare_inputs.ipynb` (shared inputs). **Writes:** `results/MCI/<model>/predictions.csv`,
`tuning.json`, `run_info.json`.

## Configuration

In [ ]:
ENDPOINT = "MCI"
import os, sys, json, random, copy, warnings, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from torch import nn
import torchtuples as tt
from pycox.models import CoxPH
from lifelines import CoxPHFitter
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from IPython.display import display

PROJECT_DIR = Path(os.environ.get("SURVIVAL_PROJECT_DIR", "/Users/yihan/Documents/Github/survival_Anlysis_Clustering"))
if str(PROJECT_DIR / "code") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "code"))
import landmark_common as lc
from landmark_common import *  # study constants and metric functions shared by every step

RUN_MODE = lc.run_mode()  # "standard"; checks/smoke_test.py sets "smoke"
SETTINGS = lc.mode_settings(RUN_MODE)
OUTER_FOLDS, MAX_EPOCHS, PATIENCE, DDH_BURN_IN = (SETTINGS[k] for k in ["outer_folds", "max_epochs", "patience", "ddh_burn_in"])
INCLUDE_DEEPSURV = os.environ.get("SURVIVAL_INCLUDE_DEEPSURV", "1") == "1"
HISTORY_MODELS = ["Cox_ridge", "DynamicDeepHit"] + (["DeepSurv"] if INCLUDE_DEEPSURV else [])
DDH_EDGES = np.arange(6., ADMIN_HORIZON + 1., 6.)
DEVICE = "cpu"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


torch.set_num_threads(min(4, os.cpu_count() or 1))
set_seed(SEED)
print(f"{ENDPOINT} · mode={RUN_MODE} · models={HISTORY_MODELS} · outer folds={OUTER_FOLDS}")

## Landmark cohort, history tables and shared folds

Uses the participants and outer folds exported in step 1.

In [ ]:
raw, datasets, accounting, history_audit = lc.load_cohort(ENDPOINT)
reference_dev = datasets["NACC"][REFERENCE_HISTORY]
outer_splits = lc.shared_outer_folds(ENDPOINT, RUN_MODE, reference_dev, OUTER_FOLDS)
shared_config = json.loads((lc.inputs_dir(ENDPOINT, RUN_MODE) / "config.json").read_text())
display(accounting)

## Fitting-only preprocessing, Landmark Cox (ridge) and DeepSurv

In [ ]:
def feature_names(df):
    return [c for c in df if c not in ["ID", "time", "event"] and not c.startswith("__")]


def make_preprocessor(df):
    numeric = [c for c in feature_names(df) if c not in CATEGORICAL]
    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("scale", StandardScaler())])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
        ("encode", OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False))])
    return Pipeline([
        ("columns", ColumnTransformer([("numeric", numeric_pipe, numeric),
                                       ("categorical", categorical_pipe, CATEGORICAL)], sparse_threshold=0)),
        ("nonconstant", VarianceThreshold())])


from lifelines.exceptions import ConvergenceWarning, ConvergenceError

class BreslowCoxLoss(nn.Module):
    def forward(self, log_h, durations, events):
        log_h, durations, events = log_h.flatten(), durations.flatten(), events.flatten()
        order = torch.argsort(durations, descending=True)
        t, r, e = durations[order], log_h[order], events[order]
        _, counts = torch.unique_consecutive(t, return_counts=True)
        ends = torch.cumsum(counts, 0) - 1
        log_denom = torch.logcumsumexp(r, 0)[ends].repeat_interleave(counts)
        return -((r-log_denom)*e).sum() / e.sum().clamp_min(1.)


def make_network(n_features, n_outputs, cfg):
    layers, previous = [], n_features
    for width in cfg["hidden"]:
        layers += [nn.Linear(previous, width), nn.ReLU(), nn.Dropout(cfg["dropout"])]
        previous = width
    layers += [nn.Linear(previous, n_outputs)]
    return nn.Sequential(*layers)


class FittedSurvival:
    def __init__(self, kind, model, preprocessor):
        self.kind, self.model, self.preprocessor = kind, model, preprocessor
    def predict(self, data, grid=EVAL_GRID):
        x = self.preprocessor.transform(data).astype("float32")
        if self.kind == "Cox_ridge":
            xdf = pd.DataFrame(x, columns=[f"x{i}" for i in range(x.shape[1])])
            curves = self.model.predict_survival_function(xdf)
        else:
            curves = self.model.predict_surv_df(x)
        times = np.asarray(curves.index, float)
        order = np.argsort(times)
        times, values = times[order], curves.to_numpy()[order]
        indices = np.searchsorted(times, grid, side="right") - 1
        pred = np.ones((len(data), len(grid)))
        known = indices >= 0
        pred[:, known] = values[indices[known]].T
        if max(grid) > ADMIN_HORIZON:
            raise ValueError("Prediction beyond configured administrative horizon.")
        return np.clip(pred, 0., 1.)


def fit_model(kind, train, cfg, seed, validation=None, epochs=None):
    set_seed(seed)
    prep = make_preprocessor(train)
    x = prep.fit_transform(train).astype("float32")
    if not np.isfinite(x).all():
        raise ValueError("Non-finite transformed training data.")
    if kind == "Cox_ridge":
        fit_df = pd.DataFrame(x, columns=[f"x{i}" for i in range(x.shape[1])])
        fit_df["time"], fit_df["event"] = train.time.to_numpy(), train.event.to_numpy()
        model = CoxPHFitter(penalizer=cfg["penalizer"], l1_ratio=0.)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            model.fit(fit_df, duration_col="time", event_col="event", show_progress=False,
                      fit_options={"step_size": 0.5, "max_steps": 200, "precision": 1e-7})
        return FittedSurvival(kind, model, prep), 0
    if kind != "DeepSurv":
        raise ValueError(f"Unsupported tabular model: {kind}")
    y = (train.time.to_numpy("float32"), train.event.to_numpy("float32"))
    network = make_network(x.shape[1], 1, cfg)
    model = CoxPH(network, torch.optim.Adam(network.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]),
                  device=DEVICE, loss=BreslowCoxLoss())
    batch = len(train)
    val_data = None
    callbacks = []
    if validation is not None:
        xv = prep.transform(validation).astype("float32")
        yv = (validation.time.to_numpy("float32"), validation.event.to_numpy("float32"))
        val_data = (xv, yv)
        callbacks = [tt.callbacks.EarlyStopping(patience=PATIENCE)]
    log = model.fit(x, y, batch_size=batch, epochs=int(epochs or MAX_EPOCHS),
                    callbacks=callbacks, verbose=False, val_data=val_data,
                    val_batch_size=len(validation) if validation is not None else 8224,
                    num_workers=0)
    if validation is not None:
        losses = log.to_pandas()["val_loss"].to_numpy()
        if not np.isfinite(losses).any():
            raise ValueError("No finite validation loss.")
        best_epoch = int(np.nanargmin(losses)) + 1
    else:
        best_epoch = int(epochs or MAX_EPOCHS)
    if kind == "DeepSurv":
        model.compute_baseline_hazards(x, y)
    return FittedSurvival(kind, model, prep), best_epoch


def tuning_grid(kind):
    if kind == "Cox_ridge":
        grid = [{"penalizer": a} for a in [0.01, 0.1, 1.0]]
    else:
        grid = [dict(hidden=[16], dropout=0.1, lr=1e-3, weight_decay=1e-3),
                dict(hidden=[32, 16], dropout=0.2, lr=1e-3, weight_decay=1e-2)]
    return grid if SETTINGS["full_tuning_grid"] else grid[:1]


def tune_and_refit(kind, development, seed):
    counts = development.event.value_counts()
    if len(counts) < 2 or counts.min() < 5:
        raise ValueError("Too few events/non-events for stratified inner validation.")
    fit_idx, val_idx = train_test_split(np.arange(len(development)), test_size=0.25,
                                        stratify=development.event, random_state=seed)
    inner_train, inner_val = development.iloc[fit_idx], development.iloc[val_idx]
    records, candidates = [], []
    for cfg in tuning_grid(kind):
        try:
            fitted, best_epoch = fit_model(kind, inner_train, cfg, seed, validation=inner_val)
            scores, _ = evaluate(inner_val, fitted.predict(inner_val), include_concordance=False)
            objective = scores[IBS_NAME]
        except (ValueError, RuntimeError, np.linalg.LinAlgError, ConvergenceError, ConvergenceWarning) as exc:
            records.append({"config": json.dumps(cfg), "status": "failed", "reason": str(exc)})
            continue
        records.append({"config": json.dumps(cfg), "inner_" + IBS_NAME: objective, "best_epoch": best_epoch})
        if np.isfinite(objective):
            candidates.append((objective, cfg, best_epoch))
    if not candidates:
        raise ValueError(f"{kind}: no supported inner-validation IBS; reconsider fixed evaluation window.")
    _, best_cfg, best_epoch = min(candidates, key=lambda c: c[0])
    # Discard inner models; refit preprocessing and model using all development participants.
    final, _ = fit_model(kind, development, best_cfg, seed, epochs=max(1, best_epoch))
    return final, {"config": best_cfg, "epochs": best_epoch, "candidates": records,
        "inner_fit_ids": development.iloc[fit_idx].ID.tolist(),
        "inner_validation_ids": development.iloc[val_idx].ID.tolist()}

## Dynamic-DeepHit (single-event, fixed-landmark adaptation)

In [ ]:
# Registry keys are lookup metadata, never model inputs. Separate cohorts prevent ID collisions.
sequence_registry = {}
for cohort, data in raw.items():
    for pid, group in data.groupby("ID", sort=False):
        past = group.loc[group.start <= LANDMARK_MONTH].sort_values("start")
        for history, count in HISTORY_LENGTHS.items():
            chosen = past.head(1) if history == "baseline_visit" else past if count is None else past.tail(count)
            sequence_registry[(cohort, str(pid), history)] = chosen[["start"] + DYNAMIC].copy()


def selected_sequences(table):
    return [sequence_registry[(cohort, str(pid), history)]
            for pid, cohort, history in table[["ID", "__cohort", "__history"]].itertuples(index=False, name=None)]


class SequencePreprocessor:
    """Fit only on selected visits of fitting participants. Zero padding has an explicit mask."""
    def fit(self, table):
        self.static = make_preprocessor(table[BASIC])
        self.static.fit(table[BASIC])
        observed = pd.concat(selected_sequences(table), ignore_index=True)[DYNAMIC].replace([np.inf, -np.inf], np.nan)
        self.mask_informative = (observed.isna().any(axis=0) & observed.notna().any(axis=0)).to_numpy(bool)
        self.median = observed.median().fillna(0.).to_numpy(float)
        imputed = np.where(np.isfinite(observed), observed, self.median)
        self.mean = imputed.mean(axis=0)
        self.scale = imputed.std(axis=0)
        self.value_informative = self.scale >= 1e-8
        self.scale[~self.value_informative] = 1.
        return self

    def transform(self, table):
        sequences = selected_sequences(table)
        lengths = np.array([len(s) for s in sequences], dtype=int)
        if len(lengths) == 0 or (lengths < 1).any():
            raise ValueError("A sequence must contain at least one real visit.")
        width, features = int(lengths.max()), len(DYNAMIC)
        values = np.zeros((len(table), width, features), dtype=np.float32)
        observed = np.zeros_like(values, dtype=bool)
        timing = np.zeros((len(table), width, 2), dtype=np.float32)
        valid = np.arange(width)[None, :] < lengths[:, None]
        for i, s in enumerate(sequences):
            t, v = s.start.to_numpy(float), s[DYNAMIC].to_numpy(float)
            if (t > LANDMARK_MONTH).any() or np.any(np.diff(t) <= 0):
                raise ValueError("Invalid prediction-time sequence.")
            m = np.isfinite(v)
            values[i, :len(s)] = (np.where(m, v, self.median)-self.mean)/self.scale*self.value_informative
            observed[i, :len(s)] = m
            # All times refer only to the selected visits; no hidden full-history gaps.
            timing[i, :len(s), 0] = (LANDMARK_MONTH-t)/12.
            timing[i, :len(s), 1] = np.r_[0., np.diff(t)]/12.
        static = self.static.transform(table[BASIC]).astype("float32")
        x = np.concatenate([values, (~observed & valid[:, :, None] & self.mask_informative[None, None, :]).astype("float32"), timing], axis=2)
        return {"x": torch.as_tensor(x), "values": torch.as_tensor(values),
                "observed": torch.as_tensor(observed), "valid": torch.as_tensor(valid),
                "static": torch.as_tensor(static), "lengths": torch.as_tensor(lengths)}


class DynamicDeepHitNet(nn.Module):
    """Single-event, fixed-landmark adaptation: recurrent history + temporal attention + PMF.

    Retains the three Dynamic-DeepHit objectives. Differences from the original TensorFlow
    implementation: conditional residual time at month 48, deterministic next-value MSE,
    explicit masks, separate fixed predictors, no prefix augmentation or competing events.
    """
    def __init__(self, sequence_features, static_features, hidden, dropout):
        super().__init__()
        self.hidden = hidden
        self.rnn = nn.GRU(sequence_features, hidden, batch_first=True)
        self.attention = nn.Sequential(nn.Linear(hidden+sequence_features+static_features, hidden),
                                       nn.Tanh(), nn.Linear(hidden, 1))
        self.head = nn.Sequential(nn.Linear(hidden+sequence_features+static_features, hidden),
                                  nn.ReLU(), nn.Dropout(dropout),
                                  nn.Linear(hidden, len(DDH_EDGES)+1))  # includes beyond-120 tail mass
        self.next_value = nn.Linear(hidden, len(DYNAMIC))

    def forward(self, batch):
        x, static, lengths = batch["x"], batch["static"], batch["lengths"]
        n, width, _ = x.shape
        latest = x[torch.arange(n), lengths-1]
        query = torch.cat([latest, static], dim=1)
        if width > 1:
            states, _ = self.rnn(x[:, :-1])
            history_mask = torch.arange(width-1)[None, :] < (lengths-1)[:, None]
            logits = self.attention(torch.cat([states, query[:, None].expand(-1, width-1, -1)], dim=2)).squeeze(-1)
            # One-visit rows have zero context, not a softmax over fictitious visits.
            weights = torch.softmax(logits.masked_fill(~history_mask, -1e9), dim=1)*history_mask
            weights = weights/weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
            context = (states*weights[:, :, None]).sum(dim=1)
            next_values = self.next_value(states)
        else:
            context = x.new_zeros((n, self.hidden))
            next_values = x.new_zeros((n, 0, len(DYNAMIC)))
            weights = x.new_zeros((n, 0))
        return self.head(torch.cat([query, context], dim=1)), next_values, weights


def ddh_survival_loss(logits, time, event, rank_weight, sigma=.1):
    """Events round up to bin endpoints; censoring rounds down to completed endpoints."""
    edges = torch.as_tensor(DDH_EDGES, dtype=time.dtype)
    event_index = torch.searchsorted(edges, time.contiguous(), right=False).clamp(max=len(DDH_EDGES)-1)
    completed = torch.searchsorted(edges, time.contiguous(), right=True)
    logp = torch.log_softmax(logits, dim=1)
    positions = torch.arange(logp.shape[1])[None, :]
    log_survival = torch.logsumexp(logp.masked_fill(positions < completed[:, None], -torch.inf), dim=1)
    log_event = logp[torch.arange(len(time)), event_index]
    nll = -torch.where(event.bool(), log_event, log_survival).mean()
    # Rank only unambiguously ordered bins; no ordering imposed on tied event bins.
    observed_bin = torch.where(event.bool(), event_index, completed-1)
    # Exact factorisation of the pairwise exponential ranking loss by event bin.
    # exp(-(F_i(b)-F_j(b))/sigma) = exp(-F_i(b)/sigma)*exp(F_j(b)/sigma).
    # This sums the same admissible pairs without materialising an n-by-n matrix.
    cdf = logp.exp().cumsum(dim=1)[:, :len(DDH_EDGES)]
    bins = torch.arange(len(DDH_EDGES))[None, :]
    event_in_bin = event.bool()[:, None] & (event_index[:, None] == bins)
    later_than_bin = observed_bin[:, None] > bins
    first_sum = (torch.exp(-cdf/sigma)*event_in_bin).sum(dim=0)
    second_sum = (torch.exp(cdf/sigma)*later_than_bin).sum(dim=0)
    n_pairs = (event_in_bin.sum(dim=0)*later_than_bin.sum(dim=0)).sum()
    ranking = (first_sum*second_sum).sum()/n_pairs.clamp_min(1)
    return nll + rank_weight*ranking

def ddh_auxiliary(next_values, batch):
    if next_values.shape[1] == 0:
        return batch["x"].sum()*0.
    target = batch["values"][:, 1:]
    mask = batch["observed"][:, 1:] & batch["valid"][:, 1:, None]
    return ((next_values-target).square()*mask).sum()/mask.sum().clamp_min(1)


class FittedDDH:
    def __init__(self, net, preprocessor):
        self.net, self.preprocessor = net, preprocessor
    def predict(self, table, grid=EVAL_GRID):
        grid = np.asarray(grid, float)
        if (grid < 0).any() or (grid > ADMIN_HORIZON).any():
            raise ValueError("Unsupported prediction horizon.")
        self.net.eval()
        with torch.no_grad():
            logits, _, _ = self.net(self.preprocessor.transform(table))
            pmf = logits.softmax(dim=1).numpy()
        # At time zero S=1. At a boundary include events in that interval.
        survival = np.column_stack([np.ones(len(table)), 1-pmf[:, :-1].cumsum(axis=1)])
        return np.clip(survival[:, np.searchsorted(DDH_EDGES, grid, side="right")], 0., 1.)


def fit_ddh(train, cfg, seed, validation=None, epochs=None):
    set_seed(seed)
    prep = SequencePreprocessor().fit(train)
    batch = prep.transform(train)
    net = DynamicDeepHitNet(batch["x"].shape[-1], batch["static"].shape[-1], cfg["width"], cfg["dropout"])
    optimizer = torch.optim.Adam(net.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    t, e = torch.tensor(train.time.to_numpy(), dtype=torch.float32), torch.tensor(train.event.to_numpy(), dtype=torch.float32)
    val_batch = prep.transform(validation) if validation is not None else None
    if validation is not None:
        vt = torch.tensor(validation.time.to_numpy(), dtype=torch.float32)
        ve = torch.tensor(validation.event.to_numpy(), dtype=torch.float32)
    # Auxiliary warmup uses only pairs of pre-landmark measurements in the fitting set.
    if batch["x"].shape[1] > 1:
        for _ in range(DDH_BURN_IN):
            optimizer.zero_grad()
            _, nxt, _ = net(batch)
            loss = ddh_auxiliary(nxt, batch)
            if loss.requires_grad:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 5.)
                optimizer.step()
    best_loss, best_state, best_epoch, stale = np.inf, None, 0, 0
    for epoch in range(1, int(epochs or MAX_EPOCHS)+1):
        net.train()
        optimizer.zero_grad()
        logits, nxt, _ = net(batch)
        loss = ddh_survival_loss(logits, t, e, cfg["rank_weight"]) + cfg["aux_weight"]*ddh_auxiliary(nxt, batch)
        if not torch.isfinite(loss):
            raise ValueError("Non-finite Dynamic-DeepHit training loss.")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.)
        optimizer.step()
        if validation is not None:
            net.eval()
            with torch.no_grad():
                vl, vn, _ = net(val_batch)
                vloss = float(ddh_survival_loss(vl, vt, ve, cfg["rank_weight"])+cfg["aux_weight"]*ddh_auxiliary(vn, val_batch))
            if np.isfinite(vloss) and vloss < best_loss-1e-7:
                best_loss, best_state, best_epoch, stale = vloss, copy.deepcopy(net.state_dict()), epoch, 0
            else:
                stale += 1
            if stale >= PATIENCE:
                break
    if validation is not None:
        if best_state is None:
            raise ValueError("No finite Dynamic-DeepHit validation loss.")
        net.load_state_dict(best_state)
    else:
        best_epoch = int(epochs or MAX_EPOCHS)
    return FittedDDH(net, prep), best_epoch


_fit_tabular, _tabular_grid = fit_model, tuning_grid


def fit_model(kind, train, cfg, seed, validation=None, epochs=None):
    if kind == "DynamicDeepHit":
        return fit_ddh(train, cfg, seed, validation, epochs)
    return _fit_tabular(kind, train, cfg, seed, validation, epochs)


def tuning_grid(kind):
    if kind == "DynamicDeepHit":
        grid = [dict(width=16, dropout=.1, lr=.001, weight_decay=.001, rank_weight=.05, aux_weight=.1),
                dict(width=32, dropout=.2, lr=.001, weight_decay=.01, rank_weight=.1, aux_weight=.1)]
        return grid if SETTINGS["full_tuning_grid"] else grid[:1]
    return _tabular_grid(kind)

## Permutation feature importance (supplementary)

For the primary condition (latest 3 visits), each outer-fold model is evaluated on its held-out NACC fold
with one feature group shuffled across participants: a baseline feature, one repeated measure (Landmark Cox/DeepSurv:
its latest value, slope and mean deviation together; Dynamic-DeepHit: its values at all three visits), or visit timing.
Importance = drop in AUC at 60 months and increase in IBS, averaged over repeats and folds. The same procedure is used
for all three models, so their importance profiles are directly comparable.

In [ ]:
IMPORTANCE_HISTORY = PRIMARY_HISTORY
IMPORTANCE_REPEATS = SETTINGS["importance_repeats"]
FEATURE_GROUPS = {**{b: [b] for b in BASIC}, **{m: [m] for m in DYNAMIC}, "visit_timing": []}
importance_rows = []


def ddh_predict_batch(fit, batch, grid=EVAL_GRID):
    """FittedDDH.predict on an already-transformed batch."""
    fit.net.eval()
    with torch.no_grad():
        logits, _, _ = fit.net(batch)
        pmf = logits.softmax(dim=1).numpy()
    survival = np.column_stack([np.ones(len(pmf)), 1 - pmf[:, :-1].cumsum(axis=1)])
    return np.clip(survival[:, np.searchsorted(DDH_EDGES, np.asarray(grid, float), side="right")], 0., 1.)


def permuted_prediction(kind, fit, table, group, perm):
    if kind != "DynamicDeepHit" or group in BASIC:
        shuffled = table.copy()
        if group == "visit_timing":
            cols = ["months_since_last_measurement"]
        elif group in BASIC:
            cols = [group]
        else:
            cols = [c for c in table.columns if c == group or c.startswith(group + "__")]
        shuffled[cols] = shuffled[cols].to_numpy()[perm]
        return fit.predict(shuffled)  # Dynamic-DeepHit reads baseline features from the table
    batch = fit.preprocessor.transform(table)
    lengths = batch["lengths"].numpy()
    assert (lengths == lengths[0]).all(), "Row-wise permutation needs equal sequence lengths."
    f = len(DYNAMIC)
    channels = [f * 2, f * 2 + 1] if group == "visit_timing" else [DYNAMIC.index(group), f + DYNAMIC.index(group)]
    x = batch["x"].clone()
    x[:, :, channels] = x[perm][:, :, channels]
    return ddh_predict_batch(fit, {**batch, "x": x})


def permutation_importance(kind, fit, table, fold):
    base, _ = evaluate(table, fit.predict(table), include_concordance=False)
    rng = np.random.default_rng(SEED + 500 + fold)
    for group in FEATURE_GROUPS:
        for repeat in range(IMPORTANCE_REPEATS):
            perm = rng.permutation(len(table))
            score, _ = evaluate(table, permuted_prediction(kind, fit, table, group, perm), include_concordance=False)
            importance_rows.append(dict(model=kind, history=IMPORTANCE_HISTORY, fold=fold, repeat=repeat, feature=group,
                                        auc60_drop=base["AUC_60"] - score["AUC_60"],
                                        ibs_increase=score[IBS_NAME] - base[IBS_NAME]))

## Fit per training set and write predictions

For every model × history: three NACC outer folds (predictions for test **and** training participants;
training predictions set the exploratory risk threshold) and the final full-NACC model (training = NACC,
test = AIBL). Each model's file is written as soon as the model finishes.

In [ ]:
jobs = [(h, k) for k in HISTORY_MODELS for h in HISTORY_LENGTHS] + [("basic_only", "Cox_ridge")]
tuning_records, inner_assignment_reference = [], {}
parts = {k: [] for k in HISTORY_MODELS}


def record_tuning(history, model, fit_label, info):
    assignment = (info["inner_fit_ids"], info["inner_validation_ids"])
    if fit_label in inner_assignment_reference:
        assert assignment == inner_assignment_reference[fit_label], "Inner splits differ across models/histories."
    else:
        inner_assignment_reference[fit_label] = assignment
    tuning_records.append({"history": history, "model": model, "fit": fit_label, **info})


def job_tables(history):
    if history == "basic_only":
        cols = ["ID", "time", "event", "__cohort", "__history"] + BASIC
        return datasets["NACC"][REFERENCE_HISTORY][cols], datasets["AIBL"][REFERENCE_HISTORY][cols]
    return datasets["NACC"][history], datasets["AIBL"][history]


def write_model(kind):
    out = lc.ensure(lc.model_dir(ENDPOINT, RUN_MODE, kind))
    pd.concat(parts[kind], ignore_index=True).to_csv(out / "predictions.csv", index=False)
    (out / "tuning.json").write_text(json.dumps([r for r in tuning_records if r["model"] == kind], indent=2))
    pd.DataFrame([r for r in importance_rows if r["model"] == kind]).to_csv(out / "feature_importance.csv", index=False)
    packages = ["torch", "pycox", "torchtuples", "lifelines", "scikit-survival", "scikit-learn", "pandas", "numpy"]
    info = dict(model=kind, endpoint=ENDPOINT, run_mode=RUN_MODE, created=datetime.now(timezone.utc).isoformat(),
                shared_config=shared_config, settings=dict(SETTINGS, seed=SEED),
                versions={p: importlib.metadata.version(p) for p in packages})
    (out / "run_info.json").write_text(json.dumps(info, indent=2, default=str))
    print(f"   -> {out / 'predictions.csv'}", flush=True)


for history, kind in jobs:
    development, external = job_tables(history)
    print(f"{history} / {kind}: outer folds", flush=True)
    for fold, (tr, te) in enumerate(outer_splits, 1):
        training = development.iloc[tr].reset_index(drop=True)
        fit, info = tune_and_refit(kind, training, SEED + fold)
        record_tuning(history, kind, f"outer_{fold}", info)
        parts[kind].append(lc.prediction_frame(development.ID.iloc[te], "NACC", f"outer_{fold}", "test", kind, history,
                                               fit.predict(development.iloc[te])))
        parts[kind].append(lc.prediction_frame(training.ID, "NACC", f"outer_{fold}", "train", kind, history,
                                               fit.predict(training)))
        if history == IMPORTANCE_HISTORY:
            permutation_importance(kind, fit, development.iloc[te].reset_index(drop=True), fold)
    print(f"{history} / {kind}: full NACC refit and AIBL prediction", flush=True)
    final, info = tune_and_refit(kind, development, SEED + 100)
    record_tuning(history, kind, "full_NACC", info)
    parts[kind].append(lc.prediction_frame(development.ID, "NACC", "full_NACC", "train", kind, history,
                                           final.predict(development)))
    parts[kind].append(lc.prediction_frame(external.ID, "AIBL", "full_NACC", "test", kind, history,
                                           final.predict(external)))
    last_job_for_model = [j for j in jobs if j[1] == kind][-1] == (history, kind)
    if last_job_for_model:
        write_model(kind)
print("Done. Next: run 04_evaluate_and_plot.ipynb.")